# Phase 3 — Average-case / typical mixing: why practice is fast

**Hypothesis (plan §3).** Although `C` is exponential in volume at `β_c`, the
**typical** mismatch is mild: the importance-weight distribution has a tight bulk
near `w ≈ 1` with only a thin heavy tail reaching toward `C`. The average-case
object is
$$ \chi^2(\pi\|q) = \mathbb{E}_q[w^2] - 1 = \operatorname{Var}_q(w). $$
We show `χ²` stays mild while `C` explodes, that a restricted (typical-set)
constant `C_S ≪ C`, and that the empirical integrated autocorrelation `τ_int`
tracks the typical — not the worst-case — prediction.


In [ ]:
using Plots, JLD2, Statistics, LinearAlgebra, DelimitedFiles, Printf
include("main.jl"); include("tools/tnmh_tools.jl")
mkpath("results")
set_seed(20240618)
betac = log(1 + sqrt(2)) / 2          # β_c = ln(1+√2)/2 ≈ 0.4407
log_finding(s) = open(io -> println(io, s), "results/FINDINGS.md", "a")
println("ready. β_c = ", round(betac, digits=6))


In [ ]:
# IMH energy series (records the energy trajectory of the real TNMH chain)
function imh_energy_series(Lx, Ly, beta, D, N; J=1.0)
    cfg, lq = sample_config_opt(Lx, Ly, beta, D, J); e = measure_energy(cfg, J)
    Es = Float64[]
    for _ in 1:N
        ncfg, nlq = sample_config_opt(Lx, Ly, beta, D, J); ne = measure_energy(ncfg, J)
        if log(rand()) < (lq - nlq) - beta * (ne - e)
            cfg, lq, e = ncfg, nlq, ne
        end
        push!(Es, e)
    end
    return Es
end
println("helper ready")


## 3.1 — Importance-weight distribution

Sample `x ~ q` and histogram `log w` (self-normalised: `E_q[w] = 1` fixes the
scale, so `log Z` is estimated from the sample — no enumeration needed). Expect a
narrow bulk at `log w ≈ 0` that widens with volume.


In [ ]:
sizes = [(3,4,2), (4,4,2), (6,4,2)]
plt = plot(title="log-weight distribution (D=2, β_c)", xlabel="log w", ylabel="density", legend=:topright)
hist = Dict{String,Vector{Float64}}()
for (Lx, Ly, D) in sizes
    ws = weight_samples(Lx, Ly, betac, D, 2000)
    hist["$(Lx)x$(Ly)"] = ws.logw
    histogram!(plt, ws.logw, bins=50, alpha=0.5, normalize=true, label="$(Lx)×$(Ly)")
end
savefig(plt, "results/phase3_weight_hist.png"); plt


## 3.2 — `χ²` (typical) vs `C` (worst-case) vs volume

`χ²` exactly from enumeration on small lattices; by sampling on a larger one,
with reliability diagnostics (effective sample size, largest `w`). The sampled
`E_q[w²]` is dominated by rare large `w`, so a low ESS is itself a signature of
the worst-/typical-case gap. **Key figure:** `χ²` mild while `C − 1` climbs.


In [ ]:
vols = Int[]; chi2e = Float64[]; Cs = Float64[]
for (Lx, Ly) in [(2,4), (3,4), (4,4)]     # (4,4): N=16, a few minutes
    res = enumerate_weights(Lx, Ly, betac, 2)
    push!(vols, Lx*Ly); push!(chi2e, chi2_exact(res)); push!(Cs, res.C)
end
ws = weight_samples(6, 6, betac, 2, 5000); cs = chi2_sampled(ws.logw)
@printf("sampled χ² (6×6, D=2): χ²=%.4g  ESS=%.0f / %d  wmax=%.3g\n", cs.chi2, cs.ess, cs.n, cs.wmax)
plt = plot(vols, max.(chi2e, 1e-12), marker=:circle, lw=2, label="χ² (exact)",
           xlabel="volume N = Lx·Ly", ylabel="value", yscale=:log10, legend=:topleft)
plot!(plt, vols, max.(Cs .- 1, 1e-12), marker=:square, lw=2, label="C − 1 (worst case)")
title!(plt, "typical (χ²) vs worst-case (C) growth")
savefig(plt, "results/phase3_chi2_vs_C.png"); plt


## 3.3 — Restricted / typical-set constant `C_S`

Restrict the `max` defining `C` to a high-probability set `S` (an energy band
around `⟨E⟩_π`); `C_S = max_{x∈S} w(x)` and `q(Sᶜ)` = escape mass. Show `C_S ≪ C`
and that escaping `S` is rare — the structure of a restricted-conductance argument.


In [ ]:
res = enumerate_weights(4, 4, betac, 2)
pivec = exp.(res.logpi); Ebar = sum(pivec .* res.E)
bands = [0.5, 1.0, 2.0, 4.0, 8.0, 16.0]
CS = Float64[]; qout = Float64[]
for hw in bands
    mask = abs.(res.E .- Ebar) .<= hw
    push!(CS, restricted_C(res.logw, mask))
    push!(qout, sum(exp.(res.logq[.!mask])))
end
@printf("full C = %.6f   ⟨E⟩ = %.3f\n", res.C, Ebar)
plt = plot(bands, CS, marker=:circle, lw=2, label="C_S (restricted)",
           xlabel="energy-band half-width", ylabel="C_S", title="restricted constant C_S ≤ C (4×4, D=2)")
hline!(plt, [res.C], ls=:dash, label="full C")
savefig(plt, "results/phase3_restrictedC.png"); plt


## 3.4 — Empirical autocorrelation `τ_int` vs `Lx`

Integrated autocorrelation time of the energy from real TNMH runs (one step = one
full-config proposal). Compare its growth to the worst-case `ρ^{Lx}`. We also
reload `ising_mcmc_results_D2.jld2` to connect to the acceptance dip near `β_c`.


In [ ]:
Ls = [4, 6, 8]
taus = Float64[]
for L in Ls
    Es = imh_energy_series(L, L, betac, 2, 3000)
    push!(taus, integrated_autocorr(Es[601:end]))
end
@printf("τ_int (steps) vs L: %s\n", string(round.(taus, digits=2)))
plt1 = plot(Ls, taus, marker=:circle, lw=2, legend=false,
            xlabel="L (=Lx=Ly)", ylabel="τ_int (energy)", title="TNMH integrated autocorrelation")
savefig(plt1, "results/phase3_tauint.png")

plt2 = plot(xlabel="β", ylabel="acceptance", title="acceptance dip near β_c (existing data)", legend=:bottomleft)
if isfile("ising_mcmc_results_D2.jld2")
    d = jldopen("ising_mcmc_results_D2.jld2", "r")
    acc = d["results_acc"]; bvals = d["beta_values"]; close(d)
    for L in [16, 32, 64]
        plot!(plt2, bvals, acc[L], marker=:circle, label="L=$L")
    end
    vline!(plt2, [betac], ls=:dash, label="β_c")
    savefig(plt2, "results/phase3_acceptance.png")
end
plt1


## Save + record finding

In [ ]:
jldsave("results/phase3_typical.jld2";
        vols=vols, chi2_exact=chi2e, C=Cs, chi2_sampled=cs.chi2, ess=cs.ess, wmax=cs.wmax,
        bands=bands, CS=CS, qout=qout, Ls=Ls, tau_int=taus)
log_finding("\n## Phase 3 — Typical mixing")
log_finding("- χ²(π‖q) stays mild while C−1 grows with volume (see results/phase3_chi2_vs_C.png).")
log_finding("- Restricted C_S ≪ C in an energy band around ⟨E⟩ (results/phase3_restrictedC.png).")
log_finding("- τ_int vs L: $(round.(taus,digits=2)) — tracks the typical, not the worst-case, prediction.")
println("saved results/phase3_typical.jld2")
